# Multi-wavelength photonic-lantern propagation

Propagates a batch of DM-aberrated input fields through the lantern
at every wavelength of a native grid (one characterized `ChainPropagator`
per wavelength, built and released one at a time by
`multi_wvl.MultiWavelengthPropagationPipeline`), giving each output fiber's
spectrum for each field. The spectra are then dispersed onto a simulated
spectrograph detector and extracted back (`spectral` package).

The aberration here is a ramp of DM mode 0 (a petal piston) from -8 lambda to
+8 lambda. Each wavelength needs its own lantern characterization: minutes per
wavelength the first time, cached under `./data/` afterwards. The
propagation results are cached too (`RESULTS_FILE`), so later runs of this
notebook only redo the plots unless `RERUN = True`.

The lantern is the preset named by the environment variable `CBEAM_LANTERN`
(`pl19_800`, the reference 19-port lantern at 800 nm, by default; or
`pl7_1200`, `pl19_1200` -- see `batch_pipeline.LANTERNS`). Executed copies
for each lantern are in `runs/<lantern>/`.


In [ ]:
import os
os.environ["CBEAM_BACKEND"] = "numpy"

import numpy as np
import matplotlib.pyplot as plt

import batch_pipeline as bp
import multi_wvl as mw
from multi_wavelength_vis import (
    plot_spectra_grid, plot_fiber_spectra, plot_wavelength_fiber_heatmap,
    plot_ramp_response, plot_total_throughput_vs_amplitude,
    plot_per_fiber_throughput_vs_amplitude, plot_detector_stage_for_field,
)

base_params = bp.get_simulation_parameters()   # the lantern named by CBEAM_LANTERN
LANTERN = base_params["lantern"]
WL_MIN_NM, WL_MAX_NM = base_params["band_nm"]
WVL_SAMPLES    = 41                    # native wavelengths (5 nm steps)
PISTON_SAMPLES = 81                    # steps of the mode-0 ramp
PISTON_MAX_NM  = 8 * base_params["wavelength_nm"]   # ramp spans -PISTON_MAX_NM .. +PISTON_MAX_NM (8 lambda)

RESULTS_FILE = f"data/mw_results_{LANTERN}_{WVL_SAMPLES}.npz"
RERUN = False                          # True: propagate again even if RESULTS_FILE exists

## Optional: mode bookkeeping across the band

Which modes are degenerate (and must be tracked as groups) and which one is
not guided changes with wavelength. `diagnose_mode_bookkeeping` infers it
from two FEM solves at the lantern ends, without characterizing anything;
`MultiWavelengthPropagationPipeline(..., auto_mode_bookkeeping=True)` does
the same for every wavelength it has to characterize from scratch. The scan
costs two FEM solves per wavelength, so it is off by default. (For pl7_1200
it finds the table of `batch_pipeline.constants` at every wavelength of the
band; pl19_1200 is characterized with the inference on, as its weakest modes
merge near 1300 nm.)

In [ ]:
RUN_BOOKKEEPING_SCAN = False

native_wl_nm = np.linspace(WL_MIN_NM, WL_MAX_NM, WVL_SAMPLES)

if RUN_BOOKKEEPING_SCAN:
    lantern = mw.build_lantern_geometry(base_params)   # shared geometry, built once
    bookkeeping = {}
    for wl in native_wl_nm:
        print(f"\n### {wl:.0f} nm")
        front, back, skipped = mw.diagnose_mode_bookkeeping(base_params, wl, PL_N=lantern)
        bookkeeping[wl] = (front, back, skipped)
    ref = bookkeeping.get(base_params["wavelength_nm"])
    print("\nwavelengths whose structure differs from the design wavelength:",
          [f"{wl:.0f}" for wl, b in bookkeeping.items() if ref is not None and b != ref])

## Propagate the ramp at every native wavelength

`get_power_spectra` returns, for every wavelength, field and output fiber,
the fraction of the incident power that reaches that fiber (the input
coupling efficiency is folded in, so the spectra are comparable across
wavelengths and fields).

In [ ]:
coeff_matrix, labels = bp.create_ramp_aberration_configs(
    [0], PISTON_SAMPLES, -PISTON_MAX_NM, PISTON_MAX_NM)
amplitudes = coeff_matrix[:, 0]

if os.path.exists(RESULTS_FILE) and not RERUN:
    data = np.load(RESULTS_FILE)
    power_spectra, wl_out = data["power_spectra"], data["wavelengths"]
    print(f"loaded {RESULTS_FILE}")
else:
    ifunc = bp.load_ifunc(base_params["ifunc_file"])   # through specula if installed
    bp.center_ifunc_mask(ifunc)
    mwp = mw.MultiWavelengthPropagationPipeline(
        base_params=base_params,
        ifunc=ifunc,
        native_wavelengths_nm=native_wl_nm,
        verbose=False,
    )
    power_spectra, wl_out = mwp.get_power_spectra(coeff_matrix, prop_chunk_size=32)
    np.savez(RESULTS_FILE, power_spectra=power_spectra, wavelengths=wl_out,
             coeff_matrix=coeff_matrix, labels=labels)
    print(f"saved {RESULTS_FILE}")

print(f"power_spectra: {power_spectra.shape} (wavelengths, fields, fibers)")

## Output spectra

For the unaberrated field (the middle of the ramp), then along the ramp.

In [ ]:
field_idx = PISTON_SAMPLES // 2          # zero piston

plot_spectra_grid(power_spectra, wl_out, field_idx=field_idx)
plot_fiber_spectra(power_spectra, wl_out, field_idx=field_idx,
                   fiber_indices=list(range(base_params["n_output_positions"])), plot_sum=True)
plot_wavelength_fiber_heatmap(power_spectra, wl_out, field_idx=field_idx)

In [ ]:
plot_ramp_response(power_spectra, wl_out, amplitudes, fiber_idx=0, labels=labels)
plot_total_throughput_vs_amplitude(power_spectra, amplitudes, labels=labels, mode_label="mode 0")
plot_per_fiber_throughput_vs_amplitude(power_spectra, amplitudes, mode_label="mode 0")

In [ ]:
# Central output core: one curve per wavelength along the ramp
central_core = 0
fig, ax = plt.subplots(figsize=(10, 6))
norm = plt.Normalize(wl_out.min(), wl_out.max())
for w, wl in enumerate(wl_out):
    ax.plot(amplitudes, power_spectra[w, :, central_core],
            color=plt.cm.viridis(norm(wl)), lw=1.0)
ax.set_xlabel("Mode-0 amplitude (nm)")
ax.set_ylabel("Fraction of incident power")
ax.set_title(f"Central output core (fiber {central_core}), one curve per wavelength")
ax.grid(True, alpha=0.3)
fig.colorbar(plt.cm.ScalarMappable(cmap="viridis", norm=norm), ax=ax, label="Wavelength (nm)")
plt.tight_layout()
plt.show()

## Input coupling across the band

Summed over the fibers, the unaberrated field's spectrum is the fraction of
the incident power that couples into the lantern's tracked modes (propagation
is lossless apart from the one untracked mode), so it shows how the coupling changes with wavelength as
the PSF scales with lambda relative to the fixed input face.

In [ ]:
coupled = power_spectra[:, field_idx, :].sum(axis=1)
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(wl_out, coupled, "o-")
ax.set_xlabel("Wavelength (nm)")
ax.set_ylabel("Coupled fraction of incident power")
ax.set_title("Unaberrated field: input coupling vs. wavelength")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
for wl, c in list(zip(wl_out, coupled))[::5]:
    print(f"  {wl:5.0f} nm   {c:6.1%}")

## Spectrograph detector stage

The unaberrated field's spectra, scaled to `photons_per_nm` photons entering
the lantern per nm of bandwidth, dispersed onto the simulated detector
(with noise) and extracted back with optimal extraction.

In [ ]:
image, extracted, variance = plot_detector_stage_for_field(
    power_spectra, wl_out, field_idx=field_idx, photons_per_nm=1e5)